In [27]:
#user profile and data models

from dataclasses import dataclass, field
from enum import Enum
from typing import List, Optional
from datetime import date

class EmploymentType(Enum):
    SALARIED_PRIVATE = "salaried_private"
    SALARIED_GOVT = "salaried_govt"
    SELF_EMPLOYED = "self_employed"
    BUSINESS_OWNER = "business_owner"
    FREELANCER = "freelancer"
    STUDENT = "student"
    RETIRED = "retired"
    UNEMPLOYED = "unemployed"

class GoalType(Enum):
    EMERGENCY_FUND = "emergency_fund"
    HOME_PURCHASE = "home_purchase"
    VEHICLE = "vehicle"
    EDUCATION_SELF = "education_self"
    EDUCATION_CHILD = "education_child"
    WEDDING = "wedding"
    TRAVEL = "travel"
    RETIREMENT = "retirement"
    WEALTH_CREATION = "wealth_creation"  # no specific target, just grow
    OTHER = "other"

class GoalPriority(Enum):
    CRITICAL = "critical"       # non-negotiable. retirement, emergency fund
    HIGH = "high"               # important but some flexibility
    ASPIRATIONAL = "aspirational"  # nice to have, first to sacrifice

@dataclass
class Goal:
    goal_type: GoalType
    priority: GoalPriority
    target_amount: float          # in rupees
    target_date: date             # when do they need this money
    current_savings_for_goal: float = 0.0  # already saved towards this
    notes: str = ""

    def years_to_goal(self) -> float:
        today = date.today()
        delta = self.target_date - today
        return delta.days / 365.25

    def amount_remaining(self) -> float:
        return self.target_amount - self.current_savings_for_goal

In [28]:
class EmploymentType(Enum):
    SALARIED_PRIVATE = "salaried_private"
    SALARIED_GOVT = "salaried_govt"
    SELF_EMPLOYED = "self_employed"
    BUSINESS_OWNER = "business_owner"
    FREELANCER = "freelancer"
    STUDENT = "student"
    RETIRED = "retired"
    UNEMPLOYED = "unemployed"

In [29]:
class GoalType(Enum):
    EMERGENCY_FUND = "emergency_fund"
    HOME_PURCHASE = "home_purchase"
    VEHICLE = "vehicle"
    EDUCATION_SELF = "education_self"
    EDUCATION_CHILD = "education_child"
    WEDDING = "wedding"
    TRAVEL = "travel"
    RETIREMENT = "retirement"
    WEALTH_CREATION = "wealth_creation"  # no specific target, just grow
    OTHER = "other"

In [30]:
class GoalPriority(Enum):
    CRITICAL = "critical"       # non-negotiable. retirement, emergency fund
    HIGH = "high"               # important but some flexibility
    ASPIRATIONAL = "aspirational"  # nice to have, first to sacrifice

In [31]:
@dataclass
class Goal:
    goal_type: GoalType
    priority: GoalPriority
    target_amount: float          # in rupees
    target_date: date             # when do they need this money
    current_savings_for_goal: float = 0.0  # already saved towards this
    notes: str = ""

    def years_to_goal(self) -> float:
        today = date.today()
        delta = self.target_date - today
        return delta.days / 365.25

    def amount_remaining(self) -> float:
        return self.target_amount - self.current_savings_for_goal

In [32]:
# test it
retirement = Goal(
    goal_type=GoalType.RETIREMENT,
    priority=GoalPriority.CRITICAL,
    target_amount=3_00_00_000,  # 3 crore
    target_date=date(2055, 1, 1),
    current_savings_for_goal=50_000
)

print(f"Years to goal: {retirement.years_to_goal():.1f}")
print(f"Amount remaining: ₹{retirement.amount_remaining():,.0f}")

Years to goal: 28.5
Amount remaining: ₹29,950,000


In [ ]:
@dataclass
class FinancialProfile:
    # Income
    monthly_income: float
    employment_type: EmploymentType
    income_stability: float        # 0.0 to 1.0. govt salary = 1.0, freelancer = 0.4

    # Expenses & obligations
    monthly_fixed_expenses: float  # rent, EMIs, insurance premiums
    monthly_variable_expenses: float  # food, transport, lifestyle

    has_secondary_income: bool = False
    secondary_monthly_income: float = 0.0

    # Existing assets
    emergency_fund_months: float = 0.0   # how many months of expenses saved
    existing_investments: float = 0.0    # current portfolio value
    provident_fund_balance: float = 0.0
    real_estate_value: float = 0.0
    gold_value: float = 0.0

    # Liabilities
    total_debt: float = 0.0
    monthly_emi_total: float = 0.0

    # Tax
    tax_bracket_percent: float = 0.0   # 0, 5, 10, 15, 20, 30 #uodate as per 2026, surcharge asw

    def monthly_surplus(self) -> float:
        return (
            self.monthly_income
            + self.secondary_monthly_income
            - self.monthly_fixed_expenses
            - self.monthly_variable_expenses
        )

    def debt_to_income_ratio(self) -> float:
        if self.monthly_income == 0:
            return 0
        return self.monthly_emi_total / self.monthly_income

    def net_worth(self) -> float:
        total_assets = (
            self.existing_investments
            + self.provident_fund_balance
            + self.real_estate_value
            + self.gold_value
        )
        return total_assets - self.total_debt

In [ ]:
class RiskToleranceLevel(Enum):
    VERY_CONSERVATIVE = 1
    CONSERVATIVE = 2
    MODERATE = 3
    AGGRESSIVE = 4
    VERY_AGGRESSIVE = 5

@dataclass
class BehavioralProfile:
    # These come from onboarding questions #still have to build onboarding questionnaire
    risk_tolerance: RiskToleranceLevel

    # How would they react if portfolio drops 20%?
    # 1 = sell everything, 5 = buy more
    drawdown_reaction_score: int   # 1 to 5

    # Investment experience
    has_invested_before: bool = False
    has_experienced_market_crash: bool = False

    # Financial literacy score (assessed during onboarding)
    financial_literacy_score: int = 1  # 1 to 5

    # Emotional relationship with money
    loss_aversion_score: int = 3   # 1 (not bothered) to 5 (very loss averse)

In [ ]:
@dataclass
class UserProfile:
    # Identity
    user_id: str
    name: str
    age: int
    date_of_birth: date
    dependents: int = 0          # number of financial dependents

    # Core profiles
    financial: FinancialProfile = None
    behavioral: BehavioralProfile = None
    goals: List[Goal] = field(default_factory=list)

    # Metadata
    profile_created: date = field(default_factory=date.today)
    last_updated: date = field(default_factory=date.today)

    def investable_surplus(self) -> float:
        """What can realistically be invested monthly"""
        surplus = self.financial.monthly_surplus()
        # Keep a buffer — don't invest 100% of surplus #hold some liquidity in cash asw
        return max(0, surplus * 0.9)

    def years_to_retirement(self, retirement_age: int = 60) -> float:
        return max(0, retirement_age - self.age)

    def life_stage(self) -> str:
        if self.age < 25:
            return "early_career"
        elif self.age < 35:
            return "wealth_building"
        elif self.age < 50:
            return "peak_earning"
        elif self.age < 60:
            return "pre_retirement"
        else:
            return "retirement"

In [36]:
# Ravi — 28, software engineer, Mumbai
ravi = UserProfile(
    user_id="u001",
    name="Ravi Shankar",
    age=28,
    date_of_birth=date(1996, 3, 15),
    dependents=0,
    financial=FinancialProfile(
        monthly_income=85000,
        employment_type=EmploymentType.SALARIED_PRIVATE,
        income_stability=0.85,
        monthly_fixed_expenses=35000,  # rent + EMI
        monthly_variable_expenses=20000,
        emergency_fund_months=1.5,
        existing_investments=80000,
        provident_fund_balance=45000,
        total_debt=200000,
        monthly_emi_total=8000,
        tax_bracket_percent=20
    ),
    behavioral=BehavioralProfile(
        risk_tolerance=RiskToleranceLevel.AGGRESSIVE,
        drawdown_reaction_score=4,
        has_invested_before=True,
        has_experienced_market_crash=False,
        financial_literacy_score=3,
        loss_aversion_score=2
    ),
    goals=[
        Goal(
            goal_type=GoalType.EMERGENCY_FUND,
            priority=GoalPriority.CRITICAL,
            target_amount=180000,   # 6 months expenses
            target_date=date(2026, 12, 31),
            current_savings_for_goal=52500  # 1.5 months already saved
        ),
        Goal(
            goal_type=GoalType.WEALTH_CREATION,
            priority=GoalPriority.HIGH,
            target_amount=1_00_00_000,
            target_date=date(2040, 1, 1)
        )
    ]
)

# Sanity checks
print(f"Monthly surplus: ₹{ravi.financial.monthly_surplus():,.0f}")
print(f"Investable surplus: ₹{ravi.investable_surplus():,.0f}")
print(f"Debt-to-income: {ravi.financial.debt_to_income_ratio():.1%}")
print(f"Net worth: ₹{ravi.financial.net_worth():,.0f}")
print(f"Life stage: {ravi.life_stage()}")
print(f"Years to retirement: {ravi.years_to_retirement():.0f}")

for goal in ravi.goals:
    print(f"\nGoal: {goal.goal_type.value}")
    print(f"  Years remaining: {goal.years_to_goal():.1f}")
    print(f"  Amount remaining: ₹{goal.amount_remaining():,.0f}")

Monthly surplus: ₹30,000
Investable surplus: ₹27,000
Debt-to-income: 9.4%
Net worth: ₹-75,000
Life stage: wealth_building
Years to retirement: 32

Goal: emergency_fund
  Years remaining: 0.5
  Amount remaining: ₹127,500

Goal: wealth_creation
  Years remaining: 13.5
  Amount remaining: ₹10,000,000


In [37]:
# risk scoring engine

import numpy as np
from dataclasses import dataclass
from enum import Enum

In [ ]:
def score_income_stability(employment_type: EmploymentType, 
                           income_stability: float) -> float:
    """
    Base score from employment type, refined by stated stability.
    Returns 0.0 to 1.0
    """
    base_scores = {
        EmploymentType.SALARIED_GOVT: 1.0,
        EmploymentType.SALARIED_PRIVATE: 0.8, #layoffs? highly replaceable
        EmploymentType.BUSINESS_OWNER: 0.6,
        EmploymentType.SELF_EMPLOYED: 0.55,
        EmploymentType.FREELANCER: 0.45,
        EmploymentType.STUDENT: 0.2,
        EmploymentType.RETIRED: 0.7,      # pension/fixed income
        EmploymentType.UNEMPLOYED: 0.0,
    }
    base = base_scores[employment_type]
    # Blend base with user-stated stability (trust both equally)
    return (base + income_stability) / 2


def score_emergency_fund(emergency_fund_months: float) -> float:
    """
    Emergency fund is the #1 prerequisite for any investment.
    Without it, any market dip forces premature withdrawal.
    Returns 0.0 to 1.0
    """
    if emergency_fund_months >= 6:
        return 1.0
    elif emergency_fund_months >= 3:
        return 0.7
    elif emergency_fund_months >= 1:
        return 0.4
    else:
        return 0.1


def score_debt_burden(debt_to_income: float) -> float:
    """
    High EMI obligations reduce investable surplus AND
    increase vulnerability to income shocks.
    Returns 0.0 to 1.0
    """
    if debt_to_income == 0:
        return 1.0
    elif debt_to_income <= 0.1:
        return 0.9
    elif debt_to_income <= 0.2:
        return 0.75
    elif debt_to_income <= 0.35:
        return 0.5
    elif debt_to_income <= 0.5:
        return 0.25
    else:
        return 0.1   # dangerously over-leveraged


def score_surplus_ratio(monthly_surplus: float, 
                        monthly_income: float) -> float:
    """
    What fraction of income is actually free to invest.
    Low surplus = low capacity even if income is high.
    Returns 0.0 to 1.0
    """
    if monthly_income == 0:
        return 0.0
    ratio = monthly_surplus / monthly_income
    if ratio >= 0.4:
        return 1.0
    elif ratio >= 0.25:
        return 0.8
    elif ratio >= 0.15:
        return 0.6
    elif ratio >= 0.05:
        return 0.35
    else:
        return 0.1


def score_dependents(dependents: int, age: int) -> float:
    """
    More dependents = more financial obligations = less capacity.
    Age matters too — dependents at 28 vs 50 are different situations.
    Returns 0.0 to 1.0
    """
    if dependents == 0:
        return 1.0
    elif dependents == 1:
        return 0.8
    elif dependents == 2:
        return 0.65
    elif dependents == 3:
        return 0.45
    else:
        return 0.3

In [39]:
def calculate_risk_capacity(profile: 'UserProfile') -> float:
    """
    Weighted combination of all capacity factors.
    Returns 0.0 to 1.0

    Weights reflect real-world importance:
    - Emergency fund is the foundation — highest weight
    - Income stability determines ability to stay invested
    - Debt burden directly competes with investments
    - Surplus ratio is the practical constraint
    - Dependents add obligation risk
    """
    f = profile.financial

    scores = {
        'emergency_fund':   score_emergency_fund(f.emergency_fund_months),
        'income_stability': score_income_stability(f.employment_type, 
                                                   f.income_stability),
        'debt_burden':      score_debt_burden(f.debt_to_income_ratio()),
        'surplus_ratio':    score_surplus_ratio(f.monthly_surplus(), 
                                                f.monthly_income),
        'dependents':       score_dependents(profile.dependents, profile.age),
    }

    weights = {
        'emergency_fund':   0.30,
        'income_stability': 0.25,
        'debt_burden':      0.20,
        'surplus_ratio':    0.15,
        'dependents':       0.10,
    }

    capacity_score = sum(scores[k] * weights[k] for k in scores)

    return round(capacity_score, 4)

In [40]:
def calculate_risk_tolerance(profile: 'UserProfile') -> float:
    """
    Behavioral/psychological risk tolerance.
    Returns 0.0 to 1.0
    """
    b = profile.behavioral

    # Stated tolerance (from enum)
    tolerance_map = {
        RiskToleranceLevel.VERY_CONSERVATIVE: 0.1,
        RiskToleranceLevel.CONSERVATIVE:      0.3,
        RiskToleranceLevel.MODERATE:          0.5,
        RiskToleranceLevel.AGGRESSIVE:        0.75,
        RiskToleranceLevel.VERY_AGGRESSIVE:   1.0,
    }
    stated = tolerance_map[b.risk_tolerance]

    # Drawdown reaction (1=panic sell, 5=buy more)
    drawdown = (b.drawdown_reaction_score - 1) / 4   # normalise to 0-1

    # Experience modifiers
    experience_bonus = 0.0
    if b.has_invested_before:
        experience_bonus += 0.05
    if b.has_experienced_market_crash:
        experience_bonus += 0.05    # survived a crash = more realistic

    # Loss aversion penalty (5=very loss averse)
    loss_aversion_penalty = (b.loss_aversion_score - 1) / 4 * 0.1

    # Financial literacy — more literate = better calibrated
    literacy_bonus = (b.financial_literacy_score - 1) / 4 * 0.05

    tolerance_score = (
        stated * 0.40
        + drawdown * 0.40
        + experience_bonus
        + literacy_bonus
        - loss_aversion_penalty
    )

    return round(min(max(tolerance_score, 0.0), 1.0), 4)  # clamp to 0-1

In [41]:
@dataclass
class RiskAssessment:
    capacity_score: float       # 0 to 1
    tolerance_score: float      # 0 to 1
    final_score: float          # 0 to 1
    final_score_100: int        # 0 to 100, for display
    risk_category: str          # human readable
    capacity_was_binding: bool  # did capacity override tolerance?
    breakdown: dict             # for explainability


def categorise_risk(score: float) -> str:
    if score < 0.2:
        return "Very Conservative"
    elif score < 0.4:
        return "Conservative"
    elif score < 0.6:
        return "Moderate"
    elif score < 0.75:
        return "Aggressive"
    else:
        return "Very Aggressive"


def assess_risk(profile: 'UserProfile') -> RiskAssessment:
    capacity = calculate_risk_capacity(profile)
    tolerance = calculate_risk_tolerance(profile)

    # Capacity acts as ceiling — can never take more risk than you can absorb
    raw_final = tolerance * 0.45 + capacity * 0.55
    final = min(raw_final, capacity)  # the ceiling

    capacity_was_binding = final < raw_final

    return RiskAssessment(
        capacity_score=capacity,
        tolerance_score=tolerance,
        final_score=round(final, 4),
        final_score_100=int(final * 100),
        risk_category=categorise_risk(final),
        capacity_was_binding=capacity_was_binding,
        breakdown={
            'capacity': capacity,
            'tolerance': tolerance,
            'raw_combined': round(raw_final, 4),
            'capacity_ceiling_applied': capacity_was_binding
        }
    )

In [42]:
ravi_risk = assess_risk(ravi)

print(f"=== Risk Assessment: {ravi.name} ===")
print(f"Capacity Score:  {ravi_risk.capacity_score:.2f} / 1.0")
print(f"Tolerance Score: {ravi_risk.tolerance_score:.2f} / 1.0")
print(f"Final Score:     {ravi_risk.final_score_100} / 100")
print(f"Risk Category:   {ravi_risk.risk_category}")
print(f"Capacity binding: {ravi_risk.capacity_was_binding}")
print(f"\nBreakdown: {ravi_risk.breakdown}")

=== Risk Assessment: Ravi Shankar ===
Capacity Score:  0.73 / 1.0
Tolerance Score: 0.65 / 1.0
Final Score:     69 / 100
Risk Category:   Aggressive
Capacity binding: False

Breakdown: {'capacity': 0.7263, 'tolerance': 0.65, 'raw_combined': 0.692, 'capacity_ceiling_applied': False}


In [43]:
# Meena — 44, tier 3 city, small business owner, 2 kids, modest savings
meena = UserProfile(
    user_id="u002",
    name="Meena Devi",
    age=44,
    date_of_birth=date(1980, 6, 20),
    dependents=2,
    financial=FinancialProfile(
        monthly_income=35000,
        employment_type=EmploymentType.BUSINESS_OWNER,
        income_stability=0.55,
        monthly_fixed_expenses=18000,
        monthly_variable_expenses=10000,
        emergency_fund_months=1.0,
        existing_investments=120000,
        provident_fund_balance=0,
        gold_value=200000,
        total_debt=150000,
        monthly_emi_total=5000,
        tax_bracket_percent=5
    ),
    behavioral=BehavioralProfile(
        risk_tolerance=RiskToleranceLevel.MODERATE,
        drawdown_reaction_score=2,
        has_invested_before=True,
        has_experienced_market_crash=False,
        financial_literacy_score=2,
        loss_aversion_score=4
    ),
    goals=[
        Goal(
            goal_type=GoalType.EDUCATION_CHILD,
            priority=GoalPriority.CRITICAL,
            target_amount=800000,
            target_date=date(2030, 6, 1),
            current_savings_for_goal=0
        ),
        Goal(
            goal_type=GoalType.RETIREMENT,
            priority=GoalPriority.HIGH,
            target_amount=1_50_00_000,
            target_date=date(2045, 1, 1)
        )
    ]
)

meena_risk = assess_risk(meena)

print(f"=== Risk Assessment: {meena.name} ===")
print(f"Capacity Score:  {meena_risk.capacity_score:.2f} / 1.0")
print(f"Tolerance Score: {meena_risk.tolerance_score:.2f} / 1.0")
print(f"Final Score:     {meena_risk.final_score_100} / 100")
print(f"Risk Category:   {meena_risk.risk_category}")
print(f"Capacity binding: {meena_risk.capacity_was_binding}")

=== Risk Assessment: Meena Devi ===
Capacity Score:  0.57 / 1.0
Tolerance Score: 0.29 / 1.0
Final Score:     44 / 100
Risk Category:   Moderate
Capacity binding: False


In [44]:
#portfolio 3 : asset alocation engine

In [45]:
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
from enum import Enum

# ── LEVEL 1: BROAD CATEGORIES ─────────────────────────────────────────
class AssetCategory(Enum):
    CASH_AND_LIQUID   = "cash_and_liquid"
    DEBT              = "debt"
    HYBRID            = "hybrid"
    EQUITY            = "equity"
    GOLD_AND_COMMODITIES = "gold_and_commodities"
    REAL_ASSETS       = "real_assets"
    TAX_ADVANTAGED    = "tax_advantaged"   # PPF, NPS, EPF — lock-in but EEE/EET


# ── LEVEL 2: SPECIFIC ASSET CLASSES ───────────────────────────────────
class AssetClass(Enum):
    # CASH & LIQUID
    SAVINGS_ACCOUNT        = "savings_account"
    LIQUID_FUND            = "liquid_fund"
    OVERNIGHT_FUND         = "overnight_fund"
    MONEY_MARKET_FUND      = "money_market_fund"

    # DEBT
    ULTRA_SHORT_FUND       = "ultra_short_fund"       # 3-6 months
    LOW_DURATION_FUND      = "low_duration_fund"       # 6-12 months
    SHORT_DURATION_FUND    = "short_duration_fund"     # 1-3 years
    CORPORATE_BOND_FUND    = "corporate_bond_fund"
    BANKING_PSU_FUND       = "banking_psu_fund"
    MEDIUM_DURATION_FUND   = "medium_duration_fund"    # 3-4 years
    GILT_FUND              = "gilt_fund"               # long duration govt
    FIXED_DEPOSIT          = "fixed_deposit"           # bank FD
    RECURRING_DEPOSIT      = "recurring_deposit"       # bank RD
    POST_OFFICE_MIS        = "post_office_mis"         # monthly income scheme
    POST_OFFICE_NSC        = "post_office_nsc"         # national savings cert
    POST_OFFICE_SCSS       = "post_office_scss"        # senior citizen scheme
    RBI_FLOATING_BOND      = "rbi_floating_bond"
    GSEC_RETAIL_DIRECT     = "gsec_retail_direct"      # RBI retail direct

    # HYBRID
    CONSERVATIVE_HYBRID    = "conservative_hybrid"    # ~25% equity
    AGGRESSIVE_HYBRID      = "aggressive_hybrid"      # ~75% equity
    BALANCED_ADVANTAGE     = "balanced_advantage"     # dynamic allocation
    ARBITRAGE_FUND         = "arbitrage_fund"         # equity taxation, debt risk

    # EQUITY
    EQUITY_INDEX_LARGECAP  = "equity_index_largecap"  # Nifty 50, Sensex
    EQUITY_LARGE_CAP       = "equity_large_cap"
    EQUITY_FLEXI_CAP       = "equity_flexi_cap"
    EQUITY_MID_CAP         = "equity_mid_cap"
    EQUITY_SMALL_CAP       = "equity_small_cap"
    EQUITY_DIVIDEND_YIELD  = "equity_dividend_yield"
    EQUITY_SECTORAL        = "equity_sectoral"
    EQUITY_ELSS            = "equity_elss"            # 80C, 3yr lock-in
    INTERNATIONAL_FUND     = "international_fund"

    # GOLD & COMMODITIES
    GOLD_ETF               = "gold_etf"
    SOVEREIGN_GOLD_BOND    = "sovereign_gold_bond"    # 8yr, 2.5% interest, zero CGT

    # REAL ASSETS
    REIT                   = "reit"
    INVIT                  = "invit"

    # TAX ADVANTAGED (separate category — lock-in changes behavior)
    PPF                    = "ppf"       # EEE, 15yr lock-in (partial withdrawal allowed)
    EPF                    = "epf"       # EEE, till retirement
    NPS_TIER1              = "nps_tier1" # EET, locked till 60
    NPS_TIER2              = "nps_tier2" # no lock-in, no extra tax benefit


# ── TAX TREATMENT ──────────────────────────────────────────────────────
class TaxTreatment(Enum):
    EEE               = "eee"          # exempt-exempt-exempt (PPF, EPF)
    EET               = "eet"          # exempt-exempt-taxed (NPS)
    EQUITY_LTCG       = "equity_ltcg"  # 10% above 1L after 1yr
    EQUITY_STCG       = "equity_stcg"  # 15% under 1yr
    DEBT_SLAB         = "debt_slab"    # added to income, taxed at slab rate
    SGB_MATURITY      = "sgb_maturity" # zero CGT at 8yr maturity; interest at slab
    REIT_INVIT        = "reit_invit"   # mixed: dividend + LTCG components
    SAVINGS_INTEREST  = "savings_interest" # slab rate, 80TTA exemption up to 10K


class DeductionBenefit(Enum):
    NONE              = "none"
    SECTION_80C       = "80c"          # up to 1.5L combined limit
    SECTION_80CCD_1B  = "80ccd_1b"    # additional 50K for NPS only
    BOTH_80C_80CCD    = "both"         # NPS Tier 1 gets both

In [46]:
@dataclass
class AssetClassProfile:
    asset_class: AssetClass
    category: AssetCategory
    
    # Returns & risk
    expected_annual_return: float      # gross, pre-tax
    annual_volatility: float
    min_horizon_years: float
    max_horizon_years: Optional[float] # None means no upper limit
    
    # Liquidity
    liquidity_score: float             # 1.0 = instant, 0.0 = fully locked
    lock_in_years: float               # 0 if no lock-in
    
    # Risk
    risk_level: int                    # 1 (safest) to 10 (riskiest)
    
    # Tax
    tax_treatment: TaxTreatment
    deduction_benefit: DeductionBenefit
    
    # Accessibility
    min_investment: float              # minimum in rupees
    sip_possible: bool                 # can invest monthly
    
    description: str

    def effective_return_after_tax(self, 
                                    tax_bracket: float,
                                    holding_years: float,
                                    has_80c_headroom: bool = False) -> float:
        """
        Estimates real after-tax return based on user's tax situation.
        tax_bracket: 0.0, 0.05, 0.10, 0.15, 0.20, 0.30
        has_80c_headroom: True if user hasn't used full 1.5L 80C limit
        """
        r = self.expected_annual_return

        if self.tax_treatment == TaxTreatment.EEE:
            return r  # no tax at any stage

        elif self.tax_treatment == TaxTreatment.EET:
            # NPS: deduction on entry, taxed on exit
            # Approximate: tax saved upfront amortised over years
            # 60% lump sum at exit is tax free, 40% annuity taxed
            annuity_tax_drag = 0.40 * tax_bracket * r
            return r - annuity_tax_drag

        elif self.tax_treatment == TaxTreatment.EQUITY_LTCG:
            if holding_years >= 1:
                # 10% on gains above 1L — approximation ignores the 1L exemption
                return r * (1 - 0.10)
            else:
                return r * (1 - 0.15)  # STCG

        elif self.tax_treatment == TaxTreatment.EQUITY_STCG:
            return r * (1 - 0.15)

        elif self.tax_treatment == TaxTreatment.DEBT_SLAB:
            return r * (1 - tax_bracket)

        elif self.tax_treatment == TaxTreatment.SGB_MATURITY:
            if holding_years >= 8:
                # interest taxed at slab, but zero CGT — very efficient
                interest_rate = 0.025
                price_appreciation = r - interest_rate
                after_tax_interest = interest_rate * (1 - tax_bracket)
                return price_appreciation + after_tax_interest
            else:
                return r * (1 - tax_bracket)

        elif self.tax_treatment == TaxTreatment.SAVINGS_INTEREST:
            # 80TTA: first 10K interest exempt, rest at slab
            return r * (1 - tax_bracket * 0.7)  # approximate

        else:
            return r * (1 - tax_bracket)

    def elss_effective_return(self,
                           tax_bracket: float,
                           monthly_investment: float,
                           has_80c_headroom: bool) -> float:
        """
        Uses IRR calculation for ELSS — the only correct way to
        model a one-time entry benefit over a multi-year holding period.
        """
        return elss_irr(
            gross_annual_return=self.expected_annual_return,
            tax_bracket=tax_bracket,
            monthly_sip=monthly_investment,
            has_80c_headroom=has_80c_headroom,
            years=3.0
        )

In [47]:
ASSET_PROFILES: Dict[AssetClass, AssetClassProfile] = {

    # ── CASH & LIQUID ────────────────────────────────────────────
    AssetClass.SAVINGS_ACCOUNT: AssetClassProfile(
        asset_class=AssetClass.SAVINGS_ACCOUNT,
        category=AssetCategory.CASH_AND_LIQUID,
        expected_annual_return=0.035,
        annual_volatility=0.0,
        min_horizon_years=0.0,
        max_horizon_years=1.0,
        liquidity_score=1.0,
        lock_in_years=0,
        risk_level=1,
        tax_treatment=TaxTreatment.SAVINGS_INTEREST,
        deduction_benefit=DeductionBenefit.NONE,
        min_investment=0,
        sip_possible=True,
        description="Regular savings account. 80TTA: first ₹10K interest exempt."
    ),
    AssetClass.LIQUID_FUND: AssetClassProfile(
        asset_class=AssetClass.LIQUID_FUND,
        category=AssetCategory.CASH_AND_LIQUID,
        expected_annual_return=0.068,
        annual_volatility=0.003,
        min_horizon_years=0.0,
        max_horizon_years=1.0,
        liquidity_score=0.95,
        lock_in_years=0,
        risk_level=1,
        tax_treatment=TaxTreatment.DEBT_SLAB,
        deduction_benefit=DeductionBenefit.NONE,
        min_investment=500,
        sip_possible=True,
        description="Liquid mutual funds. Better than savings account. T+1 redemption."
    ),
    AssetClass.OVERNIGHT_FUND: AssetClassProfile(
        asset_class=AssetClass.OVERNIGHT_FUND,
        category=AssetCategory.CASH_AND_LIQUID,
        expected_annual_return=0.065,
        annual_volatility=0.001,
        min_horizon_years=0.0,
        max_horizon_years=0.25,
        liquidity_score=1.0,
        lock_in_years=0,
        risk_level=1,
        tax_treatment=TaxTreatment.DEBT_SLAB,
        deduction_benefit=DeductionBenefit.NONE,
        min_investment=500,
        sip_possible=False,
        description="Overnight funds. Safest mutual fund category. For parking money."
    ),
    AssetClass.MONEY_MARKET_FUND: AssetClassProfile(
        asset_class=AssetClass.MONEY_MARKET_FUND,
        category=AssetCategory.CASH_AND_LIQUID,
        expected_annual_return=0.072,
        annual_volatility=0.005,
        min_horizon_years=0.25,
        max_horizon_years=1.0,
        liquidity_score=0.9,
        lock_in_years=0,
        risk_level=2,
        tax_treatment=TaxTreatment.DEBT_SLAB,
        deduction_benefit=DeductionBenefit.NONE,
        min_investment=500,
        sip_possible=True,
        description="Money market funds. Up to 1yr maturity instruments."
    ),

    # ── DEBT ─────────────────────────────────────────────────────
    AssetClass.ULTRA_SHORT_FUND: AssetClassProfile(
        asset_class=AssetClass.ULTRA_SHORT_FUND,
        category=AssetCategory.DEBT,
        expected_annual_return=0.073,
        annual_volatility=0.008,
        min_horizon_years=0.25,
        max_horizon_years=1.0,
        liquidity_score=0.9,
        lock_in_years=0,
        risk_level=2,
        tax_treatment=TaxTreatment.DEBT_SLAB,
        deduction_benefit=DeductionBenefit.NONE,
        min_investment=500,
        sip_possible=True,
        description="3-6 month horizon debt funds."
    ),
    AssetClass.LOW_DURATION_FUND: AssetClassProfile(
        asset_class=AssetClass.LOW_DURATION_FUND,
        category=AssetCategory.DEBT,
        expected_annual_return=0.076,
        annual_volatility=0.012,
        min_horizon_years=0.5,
        max_horizon_years=2.0,
        liquidity_score=0.88,
        lock_in_years=0,
        risk_level=2,
        tax_treatment=TaxTreatment.DEBT_SLAB,
        deduction_benefit=DeductionBenefit.NONE,
        min_investment=500,
        sip_possible=True,
        description="6-12 month horizon debt funds."
    ),
    AssetClass.SHORT_DURATION_FUND: AssetClassProfile(
        asset_class=AssetClass.SHORT_DURATION_FUND,
        category=AssetCategory.DEBT,
        expected_annual_return=0.078,
        annual_volatility=0.018,
        min_horizon_years=1.0,
        max_horizon_years=3.0,
        liquidity_score=0.85,
        lock_in_years=0,
        risk_level=3,
        tax_treatment=TaxTreatment.DEBT_SLAB,
        deduction_benefit=DeductionBenefit.NONE,
        min_investment=500,
        sip_possible=True,
        description="1-3 year horizon debt funds."
    ),
    AssetClass.CORPORATE_BOND_FUND: AssetClassProfile(
        asset_class=AssetClass.CORPORATE_BOND_FUND,
        category=AssetCategory.DEBT,
        expected_annual_return=0.082,
        annual_volatility=0.025,
        min_horizon_years=2.0,
        max_horizon_years=5.0,
        liquidity_score=0.82,
        lock_in_years=0,
        risk_level=3,
        tax_treatment=TaxTreatment.DEBT_SLAB,
        deduction_benefit=DeductionBenefit.NONE,
        min_investment=500,
        sip_possible=True,
        description="AA+ rated corporate bond funds. Higher yield, minimal credit risk."
    ),
    AssetClass.BANKING_PSU_FUND: AssetClassProfile(
        asset_class=AssetClass.BANKING_PSU_FUND,
        category=AssetCategory.DEBT,
        expected_annual_return=0.079,
        annual_volatility=0.020,
        min_horizon_years=1.0,
        max_horizon_years=4.0,
        liquidity_score=0.85,
        lock_in_years=0,
        risk_level=2,
        tax_treatment=TaxTreatment.DEBT_SLAB,
        deduction_benefit=DeductionBenefit.NONE,
        min_investment=500,
        sip_possible=True,
        description="Banking and PSU debt funds. Very low credit risk."
    ),
    AssetClass.MEDIUM_DURATION_FUND: AssetClassProfile(
        asset_class=AssetClass.MEDIUM_DURATION_FUND,
        category=AssetCategory.DEBT,
        expected_annual_return=0.083,
        annual_volatility=0.035,
        min_horizon_years=3.0,
        max_horizon_years=5.0,
        liquidity_score=0.80,
        lock_in_years=0,
        risk_level=3,
        tax_treatment=TaxTreatment.DEBT_SLAB,
        deduction_benefit=DeductionBenefit.NONE,
        min_investment=500,
        sip_possible=True,
        description="3-4 year Macaulay duration debt funds."
    ),
    AssetClass.GILT_FUND: AssetClassProfile(
        asset_class=AssetClass.GILT_FUND,
        category=AssetCategory.DEBT,
        expected_annual_return=0.087,
        annual_volatility=0.055,
        min_horizon_years=5.0,
        max_horizon_years=None,
        liquidity_score=0.78,
        lock_in_years=0,
        risk_level=4,
        tax_treatment=TaxTreatment.DEBT_SLAB,
        deduction_benefit=DeductionBenefit.NONE,
        min_investment=500,
        sip_possible=True,
        description="Long duration government securities. Interest rate sensitive."
    ),
    AssetClass.FIXED_DEPOSIT: AssetClassProfile(
        asset_class=AssetClass.FIXED_DEPOSIT,
        category=AssetCategory.DEBT,
        expected_annual_return=0.074,
        annual_volatility=0.0,
        min_horizon_years=0.25,
        max_horizon_years=5.0,
        liquidity_score=0.70,
        lock_in_years=0,           # technically breakable with penalty
        risk_level=1,
        tax_treatment=TaxTreatment.DEBT_SLAB,
        deduction_benefit=DeductionBenefit.NONE,
        min_investment=1000,
        sip_possible=False,
        description="Bank FDs. DICGC insured up to ₹5L. Guaranteed returns. "
                    "Premature withdrawal with penalty. Very popular in tier 2/3."
    ),
    AssetClass.RECURRING_DEPOSIT: AssetClassProfile(
        asset_class=AssetClass.RECURRING_DEPOSIT,
        category=AssetCategory.DEBT,
        expected_annual_return=0.072,
        annual_volatility=0.0,
        min_horizon_years=0.5,
        max_horizon_years=5.0,
        liquidity_score=0.65,
        lock_in_years=0,
        risk_level=1,
        tax_treatment=TaxTreatment.DEBT_SLAB,
        deduction_benefit=DeductionBenefit.NONE,
        min_investment=100,
        sip_possible=True,         # IS a SIP by nature
        description="Recurring deposit. Monthly fixed contribution. "
                    "Familiar to tier 2/3 users. Safe and guaranteed."
    ),
    AssetClass.POST_OFFICE_MIS: AssetClassProfile(
        asset_class=AssetClass.POST_OFFICE_MIS,
        category=AssetCategory.DEBT,
        expected_annual_return=0.077,
        annual_volatility=0.0,
        min_horizon_years=5.0,
        max_horizon_years=5.0,
        liquidity_score=0.50,
        lock_in_years=5,
        risk_level=1,
        tax_treatment=TaxTreatment.DEBT_SLAB,
        deduction_benefit=DeductionBenefit.NONE,
        min_investment=1000,
        sip_possible=False,
        description="Post Office Monthly Income Scheme. Govt guaranteed. "
                    "Monthly income payout. Max ₹9L single, ₹15L joint. "
                    "Extremely trusted in tier 2/3 India."
    ),
    AssetClass.POST_OFFICE_NSC: AssetClassProfile(
        asset_class=AssetClass.POST_OFFICE_NSC,
        category=AssetCategory.DEBT,
        expected_annual_return=0.077,
        annual_volatility=0.0,
        min_horizon_years=5.0,
        max_horizon_years=5.0,
        liquidity_score=0.30,
        lock_in_years=5,
        risk_level=1,
        tax_treatment=TaxTreatment.DEBT_SLAB,
        deduction_benefit=DeductionBenefit.SECTION_80C,
        min_investment=1000,
        sip_possible=False,
        description="National Savings Certificate. 80C eligible. 5yr lock-in. "
                    "Govt guaranteed. Accrued interest reinvested counts as 80C."
    ),
    AssetClass.POST_OFFICE_SCSS: AssetClassProfile(
        asset_class=AssetClass.POST_OFFICE_SCSS,
        category=AssetCategory.DEBT,
        expected_annual_return=0.082,
        annual_volatility=0.0,
        min_horizon_years=5.0,
        max_horizon_years=5.0,
        liquidity_score=0.40,
        lock_in_years=5,
        risk_level=1,
        tax_treatment=TaxTreatment.DEBT_SLAB,
        deduction_benefit=DeductionBenefit.SECTION_80C,
        min_investment=1000,
        sip_possible=False,
        description="Senior Citizen Savings Scheme. Age 60+. 8.2% rate. "
                    "80C eligible. Max ₹30L. Highest safe return for retirees."
    ),
    AssetClass.RBI_FLOATING_BOND: AssetClassProfile(
        asset_class=AssetClass.RBI_FLOATING_BOND,
        category=AssetCategory.DEBT,
        expected_annual_return=0.082,
        annual_volatility=0.0,
        min_horizon_years=7.0,
        max_horizon_years=7.0,
        liquidity_score=0.20,
        lock_in_years=7,
        risk_level=1,
        tax_treatment=TaxTreatment.DEBT_SLAB,
        deduction_benefit=DeductionBenefit.NONE,
        min_investment=1000,
        sip_possible=False,
        description="RBI Floating Rate Savings Bonds. 7yr lock-in. "
                    "Rate linked to NSC + 35bps. Sovereign guarantee."
    ),
    AssetClass.GSEC_RETAIL_DIRECT: AssetClassProfile(
        asset_class=AssetClass.GSEC_RETAIL_DIRECT,
        category=AssetCategory.DEBT,
        expected_annual_return=0.073,
        annual_volatility=0.04,
        min_horizon_years=1.0,
        max_horizon_years=None,
        liquidity_score=0.65,
        lock_in_years=0,
        risk_level=2,
        tax_treatment=TaxTreatment.DEBT_SLAB,
        deduction_benefit=DeductionBenefit.NONE,
        min_investment=10000,
        sip_possible=False,
        description="G-Secs via RBI Retail Direct. Sovereign. "
                    "Interest rate risk if sold before maturity."
    ),

    # ── HYBRID ───────────────────────────────────────────────────
    AssetClass.CONSERVATIVE_HYBRID: AssetClassProfile(
        asset_class=AssetClass.CONSERVATIVE_HYBRID,
        category=AssetCategory.HYBRID,
        expected_annual_return=0.090,
        annual_volatility=0.08,
        min_horizon_years=2.0,
        max_horizon_years=None,
        liquidity_score=0.85,
        lock_in_years=0,
        risk_level=3,
        tax_treatment=TaxTreatment.DEBT_SLAB,   # <65% equity = debt taxation
        deduction_benefit=DeductionBenefit.NONE,
        min_investment=500,
        sip_possible=True,
        description="75% debt / 25% equity. Stable with mild growth. "
                    "Good for conservative users who want slight equity kicker."
    ),
    AssetClass.AGGRESSIVE_HYBRID: AssetClassProfile(
        asset_class=AssetClass.AGGRESSIVE_HYBRID,
        category=AssetCategory.HYBRID,
        expected_annual_return=0.118,
        annual_volatility=0.155,
        min_horizon_years=3.0,
        max_horizon_years=None,
        liquidity_score=0.85,
        lock_in_years=0,
        risk_level=5,
        tax_treatment=TaxTreatment.EQUITY_LTCG,  # >65% equity = equity taxation
        deduction_benefit=DeductionBenefit.NONE,
        min_investment=500,
        sip_possible=True,
        description="75% equity / 25% debt. Equity taxation. "
                    "Good moderate-risk product."
    ),
    AssetClass.BALANCED_ADVANTAGE: AssetClassProfile(
        asset_class=AssetClass.BALANCED_ADVANTAGE,
        category=AssetCategory.HYBRID,
        expected_annual_return=0.110,
        annual_volatility=0.120,
        min_horizon_years=3.0,
        max_horizon_years=None,
        liquidity_score=0.85,
        lock_in_years=0,
        risk_level=4,
        tax_treatment=TaxTreatment.EQUITY_LTCG,
        deduction_benefit=DeductionBenefit.NONE,
        min_investment=500,
        sip_possible=True,
        description="Dynamic asset allocation. Fund manager shifts equity/debt "
                    "based on valuations. Good for market-timing-averse investors."
    ),
    AssetClass.ARBITRAGE_FUND: AssetClassProfile(
        asset_class=AssetClass.ARBITRAGE_FUND,
        category=AssetCategory.HYBRID,
        expected_annual_return=0.068,
        annual_volatility=0.005,
        min_horizon_years=0.25,
        max_horizon_years=None,
        liquidity_score=0.88,
        lock_in_years=0,
        risk_level=2,
        tax_treatment=TaxTreatment.EQUITY_LTCG,  # key advantage
        deduction_benefit=DeductionBenefit.NONE,
        min_investment=500,
        sip_possible=True,
        description="Near-zero risk but taxed as equity after 1yr holding. "
            "Rolling strategy: invest, hold 12+ months, redeem at LTCG rate. "
            "For 30%+ bracket: beats FDs and most debt funds on after-tax basis. "
            "Ideal for parking 1yr+ money for high bracket investors."
    ),

    # ── EQUITY ───────────────────────────────────────────────────
    AssetClass.EQUITY_INDEX_LARGECAP: AssetClassProfile(
        asset_class=AssetClass.EQUITY_INDEX_LARGECAP,
        category=AssetCategory.EQUITY,
        expected_annual_return=0.120,
        annual_volatility=0.175,
        min_horizon_years=5.0,
        max_horizon_years=None,
        liquidity_score=0.92,
        lock_in_years=0,
        risk_level=5,
        tax_treatment=TaxTreatment.EQUITY_LTCG,
        deduction_benefit=DeductionBenefit.NONE,
        min_investment=500,
        sip_possible=True,
        description="Nifty 50 / Sensex index funds. Lowest cost equity exposure. "
                    "Recommended as core equity holding for most investors."
    ),
    AssetClass.EQUITY_LARGE_CAP: AssetClassProfile(
        asset_class=AssetClass.EQUITY_LARGE_CAP,
        category=AssetCategory.EQUITY,
        expected_annual_return=0.122,
        annual_volatility=0.180,
        min_horizon_years=5.0,
        max_horizon_years=None,
        liquidity_score=0.90,
        lock_in_years=0,
        risk_level=5,
        tax_treatment=TaxTreatment.EQUITY_LTCG,
        deduction_benefit=DeductionBenefit.NONE,
        min_investment=500,
        sip_possible=True,
        description="Actively managed large cap funds. "
                    "Prefer index funds unless strong alpha track record."
    ),
    AssetClass.EQUITY_FLEXI_CAP: AssetClassProfile(
        asset_class=AssetClass.EQUITY_FLEXI_CAP,
        category=AssetCategory.EQUITY,
        expected_annual_return=0.130,
        annual_volatility=0.200,
        min_horizon_years=5.0,
        max_horizon_years=None,
        liquidity_score=0.88,
        lock_in_years=0,
        risk_level=6,
        tax_treatment=TaxTreatment.EQUITY_LTCG,
        deduction_benefit=DeductionBenefit.NONE,
        min_investment=500,
        sip_possible=True,
        description="Fund manager allocates freely across large/mid/small. "
                    "Good single-fund equity exposure."
    ),
    AssetClass.EQUITY_MID_CAP: AssetClassProfile(
        asset_class=AssetClass.EQUITY_MID_CAP,
        category=AssetCategory.EQUITY,
        expected_annual_return=0.145,
        annual_volatility=0.240,
        min_horizon_years=7.0,
        max_horizon_years=None,
        liquidity_score=0.85,
        lock_in_years=0,
        risk_level=7,
        tax_treatment=TaxTreatment.EQUITY_LTCG,
        deduction_benefit=DeductionBenefit.NONE,
        min_investment=500,
        sip_possible=True,
        description="Mid cap mutual funds. Higher return potential, higher volatility."
    ),
    AssetClass.EQUITY_SMALL_CAP: AssetClassProfile(
        asset_class=AssetClass.EQUITY_SMALL_CAP,
        category=AssetCategory.EQUITY,
        expected_annual_return=0.160,
        annual_volatility=0.300,
        min_horizon_years=7.0,
        max_horizon_years=None,
        liquidity_score=0.80,
        lock_in_years=0,
        risk_level=8,
        tax_treatment=TaxTreatment.EQUITY_LTCG,
        deduction_benefit=DeductionBenefit.NONE,
        min_investment=500,
        sip_possible=True,
        description="Small cap funds. High risk, high reward. "
                    "Only for long horizon aggressive investors."
    ),
    AssetClass.EQUITY_DIVIDEND_YIELD: AssetClassProfile(
        asset_class=AssetClass.EQUITY_DIVIDEND_YIELD,
        category=AssetCategory.EQUITY,
        expected_annual_return=0.115,
        annual_volatility=0.145,
        min_horizon_years=5.0,
        max_horizon_years=None,
        liquidity_score=0.88,
        lock_in_years=0,
        risk_level=4,
        tax_treatment=TaxTreatment.EQUITY_LTCG,
        deduction_benefit=DeductionBenefit.NONE,
        min_investment=500,
        sip_possible=True,
        description="Dividend yield funds. Lower volatility equity. "
                    "Good for pre-retirees wanting equity with stability."
    ),
    AssetClass.EQUITY_SECTORAL: AssetClassProfile(
        asset_class=AssetClass.EQUITY_SECTORAL,
        category=AssetCategory.EQUITY,
        expected_annual_return=0.150,
        annual_volatility=0.280,
        min_horizon_years=7.0,
        max_horizon_years=None,
        liquidity_score=0.80,
        lock_in_years=0,
        risk_level=9,
        tax_treatment=TaxTreatment.EQUITY_LTCG,
        deduction_benefit=DeductionBenefit.NONE,
        min_investment=500,
        sip_possible=True,
        description="Sectoral / thematic funds. Concentrated risk. "
                    "Only for users with strong conviction on a sector."
    ),
    AssetClass.EQUITY_ELSS: AssetClassProfile(
        asset_class=AssetClass.EQUITY_ELSS,
        category=AssetCategory.EQUITY,
        expected_annual_return=0.128,
        annual_volatility=0.195,
        min_horizon_years=3.0,
        max_horizon_years=None,
        liquidity_score=0.60,
        lock_in_years=3,
        risk_level=6,
        tax_treatment=TaxTreatment.EQUITY_LTCG,
        deduction_benefit=DeductionBenefit.SECTION_80C,
        min_investment=500,
        sip_possible=True,
        description="Equity Linked Savings Scheme. 80C deduction up to ₹1.5L. "
                    "3yr lock-in (shortest among 80C options). "
                    "Should be first equity choice if 80C headroom exists."
    ),
    AssetClass.INTERNATIONAL_FUND: AssetClassProfile(
        asset_class=AssetClass.INTERNATIONAL_FUND,
        category=AssetCategory.EQUITY,
        expected_annual_return=0.110,
        annual_volatility=0.200,
        min_horizon_years=5.0,
        max_horizon_years=None,
        liquidity_score=0.80,
        lock_in_years=0,
        risk_level=6,
        tax_treatment=TaxTreatment.DEBT_SLAB,  # taxed as debt post 2023
        deduction_benefit=DeductionBenefit.NONE,
        min_investment=500,
        sip_possible=True,
        description="US / global index funds. Currency + market diversification. "
                    "Now taxed at slab rate — reduces attractiveness for 30% bracket."
    ),

    # ── GOLD & COMMODITIES ────────────────────────────────────────
    AssetClass.GOLD_ETF: AssetClassProfile(
        asset_class=AssetClass.GOLD_ETF,
        category=AssetCategory.GOLD_AND_COMMODITIES,
        expected_annual_return=0.090,
        annual_volatility=0.150,
        min_horizon_years=3.0,
        max_horizon_years=None,
        liquidity_score=0.88,
        lock_in_years=0,
        risk_level=4,
        tax_treatment=TaxTreatment.DEBT_SLAB,   # taxed as debt post 2023
        deduction_benefit=DeductionBenefit.NONE,
        min_investment=500,
        sip_possible=True,
        description="Gold ETFs. Real-time pricing. No making charges. "
                    "Taxed at slab rate now. Better than physical gold for purity/storage."
    ),
    AssetClass.SOVEREIGN_GOLD_BOND: AssetClassProfile(
        asset_class=AssetClass.SOVEREIGN_GOLD_BOND,
        category=AssetCategory.GOLD_AND_COMMODITIES,
        expected_annual_return=0.115,   # gold price + 2.5% interest
        annual_volatility=0.150,
        min_horizon_years=5.0,
        max_horizon_years=8.0,
        liquidity_score=0.40,
        lock_in_years=5,               # early exit from yr 5, full maturity yr 8
        risk_level=4,
        tax_treatment=TaxTreatment.SGB_MATURITY,
        deduction_benefit=DeductionBenefit.NONE,
        min_investment=5000,
        sip_possible=False,
        description="Sovereign Gold Bonds. RBI issued. 2.5% annual interest. "
                    "ZERO capital gains tax at 8yr maturity. "
                    "Best gold instrument for long-term holders by a wide margin."
    ),

    # ── REAL ASSETS ──────────────────────────────────────────────
    AssetClass.REIT: AssetClassProfile(
        asset_class=AssetClass.REIT,
        category=AssetCategory.REAL_ASSETS,
        expected_annual_return=0.105,
        annual_volatility=0.160,
        min_horizon_years=5.0,
        max_horizon_years=None,
        liquidity_score=0.75,
        lock_in_years=0,
        risk_level=5,
        tax_treatment=TaxTreatment.REIT_INVIT,
        deduction_benefit=DeductionBenefit.NONE,
        min_investment=10000,
        sip_possible=False,
        description="Real Estate Investment Trusts. Regular dividend income + "
                    "capital appreciation. Nifty REITs index available. "
                    "Good for real estate exposure without buying property."
    ),
    AssetClass.INVIT: AssetClassProfile(
        asset_class=AssetClass.INVIT,
        category=AssetCategory.REAL_ASSETS,
        expected_annual_return=0.110,
        annual_volatility=0.140,
        min_horizon_years=5.0,
        max_horizon_years=None,
        liquidity_score=0.65,
        lock_in_years=0,
        risk_level=5,
        tax_treatment=TaxTreatment.REIT_INVIT,
        deduction_benefit=DeductionBenefit.NONE,
        min_investment=10000,
        sip_possible=False,
        description="Infrastructure Investment Trusts. Roads, power, telecom. "
                    "High dividend yield. Less liquid than REITs."
    ),

    # ── TAX ADVANTAGED ────────────────────────────────────────────
    AssetClass.PPF: AssetClassProfile(
        asset_class=AssetClass.PPF,
        category=AssetCategory.TAX_ADVANTAGED,
        expected_annual_return=0.071,
        annual_volatility=0.0,
        min_horizon_years=15.0,
        max_horizon_years=None,
        liquidity_score=0.25,
        lock_in_years=15,
        risk_level=1,
        tax_treatment=TaxTreatment.EEE,
        deduction_benefit=DeductionBenefit.SECTION_80C,
        min_investment=500,
        sip_possible=True,
        description="Public Provident Fund. EEE status. 80C eligible. "
                    "Govt guaranteed. 15yr lock-in with partial withdrawal from yr 7. "
                    "Max ₹1.5L/year. Best risk-free long-term instrument."
    ),
    AssetClass.EPF: AssetClassProfile(
        asset_class=AssetClass.EPF,
        category=AssetCategory.TAX_ADVANTAGED,
        expected_annual_return=0.082,
        annual_volatility=0.0,
        min_horizon_years=0.0,     # technically till retirement but we track it
        max_horizon_years=None,
        liquidity_score=0.15,
        lock_in_years=0,           # withdrawable under conditions
        risk_level=1,
        tax_treatment=TaxTreatment.EEE,
        deduction_benefit=DeductionBenefit.SECTION_80C,
        min_investment=0,
        sip_possible=False,        # employer-driven
        description="Employee Provident Fund. EEE status. Employer matches. "
                    "8.25% current rate. Counted as existing asset, not new investment."
    ),
    AssetClass.NPS_TIER1: AssetClassProfile(
        asset_class=AssetClass.NPS_TIER1,
        category=AssetCategory.TAX_ADVANTAGED,
        expected_annual_return=0.105,   # blended equity+debt NPS portfolio
        annual_volatility=0.120,
        min_horizon_years=10.0,
        max_horizon_years=None,
        liquidity_score=0.10,
        lock_in_years=0,               # locked till 60
        risk_level=4,
        tax_treatment=TaxTreatment.EET,
        deduction_benefit=DeductionBenefit.BOTH_80C_80CCD,
        min_investment=500,
        sip_possible=True,
        description="National Pension System Tier 1. Locked till age 60. "
                    "80C + additional 80CCD(1B) ₹50K deduction. "
                    "60% lump sum tax-free at exit, 40% mandatory annuity. "
                    "For 30% bracket: ₹50K extra deduction = ₹15K tax saved annually."
    ),
    AssetClass.NPS_TIER2: AssetClassProfile(
        asset_class=AssetClass.NPS_TIER2,
        category=AssetCategory.TAX_ADVANTAGED,
        expected_annual_return=0.105,
        annual_volatility=0.120,
        min_horizon_years=0.0,
        max_horizon_years=None,
        liquidity_score=0.80,
        lock_in_years=0,
        risk_level=4,
        tax_treatment=TaxTreatment.DEBT_SLAB,   # no special tax benefit
        deduction_benefit=DeductionBenefit.NONE,
        min_investment=500,
        sip_possible=True,
        description="NPS Tier 2. No lock-in, no tax benefit. "
                    "Liquid NPS with good fund management. "
                    "Only useful if you want NPS fund managers without lock-in."
    ),
}

In [48]:
# Verify all asset classes have a profile
defined = set(ASSET_PROFILES.keys())
all_classes = set(AssetClass)
missing = all_classes - defined

if missing:
    print(f"⚠️  Missing profiles for: {missing}")
else:
    print(f"✅  All {len(defined)} asset classes have profiles")

# Summary by category
df = pd.DataFrame([{
    'asset': ac.value,
    'category': ASSET_PROFILES[ac].category.value,
    'return': f"{ASSET_PROFILES[ac].expected_annual_return:.1%}",
    'volatility': f"{ASSET_PROFILES[ac].annual_volatility:.1%}",
    'risk_level': ASSET_PROFILES[ac].risk_level,
    'lock_in': ASSET_PROFILES[ac].lock_in_years,
    'tax': ASSET_PROFILES[ac].tax_treatment.value,
    'deduction': ASSET_PROFILES[ac].deduction_benefit.value,
    'min_invest': f"₹{ASSET_PROFILES[ac].min_investment:,.0f}",
} for ac in AssetClass])

print(f"\nFull asset universe ({len(df)} instruments):\n")
print(df.to_string(index=False))

✅  All 39 asset classes have profiles

Full asset universe (39 instruments):

                asset             category return volatility  risk_level  lock_in              tax deduction min_invest
      savings_account      cash_and_liquid   3.5%       0.0%           1        0 savings_interest      none         ₹0
          liquid_fund      cash_and_liquid   6.8%       0.3%           1        0        debt_slab      none       ₹500
       overnight_fund      cash_and_liquid   6.5%       0.1%           1        0        debt_slab      none       ₹500
    money_market_fund      cash_and_liquid   7.2%       0.5%           2        0        debt_slab      none       ₹500
     ultra_short_fund                 debt   7.3%       0.8%           2        0        debt_slab      none       ₹500
    low_duration_fund                 debt   7.6%       1.2%           2        0        debt_slab      none       ₹500
  short_duration_fund                 debt   7.8%       1.8%           3        0 

In [49]:
def get_tax_optimised_alternatives(
    asset_class: AssetClass,
    tax_bracket: float,
    has_80c_headroom: bool,
    has_nps_headroom: bool,
    years: float
) -> List[tuple]:
    """
    Given a generic need (e.g. 'I need debt exposure for 3 years'),
    return ranked alternatives by after-tax return for this user's
    specific tax situation.

    Returns: list of (AssetClass, effective_return, reason) sorted by return
    """
    profile = ASSET_PROFILES[asset_class]
    category = profile.category

    # Find all asset classes in same category with compatible horizon
    candidates = [
        ac for ac, p in ASSET_PROFILES.items()
        if p.category == category
        and p.min_horizon_years <= years
        and (p.max_horizon_years is None or p.max_horizon_years >= years)
    ]

    results = []
    for ac in candidates:
        p = ASSET_PROFILES[ac]
        if ac == AssetClass.EQUITY_ELSS:
            eff = p.elss_effective_return(
                tax_bracket,
                150000,
                has_80c_headroom
            )
            reason = "80C benefit included" if has_80c_headroom else "No 80C headroom"
        else:
            eff = p.effective_return_after_tax(tax_bracket, years, has_80c_headroom)
            reason = p.tax_treatment.value

        results.append((ac, round(eff, 4), reason))

    return sorted(results, key=lambda x: -x[1])


# Test: for a 30% bracket user, what's the best debt option for 3 years?
print("Best debt options for 30% bracket, 3yr horizon:")
print("─" * 60)
alternatives = get_tax_optimised_alternatives(
    AssetClass.SHORT_DURATION_FUND,
    tax_bracket=0.30,
    has_80c_headroom=True,
    has_nps_headroom=True,
    years=3
)
for ac, ret, reason in alternatives:
    print(f"  {ac.value:<30} {ret:.2%}  ({reason})")

print("\nSame for 5% bracket user:")
print("─" * 60)
alternatives_low = get_tax_optimised_alternatives(
    AssetClass.SHORT_DURATION_FUND,
    tax_bracket=0.05,
    has_80c_headroom=True,
    has_nps_headroom=True,
    years=3
)
for ac, ret, reason in alternatives_low:
    print(f"  {ac.value:<30} {ret:.2%}  ({reason})")

Best debt options for 30% bracket, 3yr horizon:
────────────────────────────────────────────────────────────
  medium_duration_fund           5.81%  (debt_slab)
  corporate_bond_fund            5.74%  (debt_slab)
  banking_psu_fund               5.53%  (debt_slab)
  short_duration_fund            5.46%  (debt_slab)
  fixed_deposit                  5.18%  (debt_slab)
  gsec_retail_direct             5.11%  (debt_slab)
  recurring_deposit              5.04%  (debt_slab)

Same for 5% bracket user:
────────────────────────────────────────────────────────────
  medium_duration_fund           7.89%  (debt_slab)
  corporate_bond_fund            7.79%  (debt_slab)
  banking_psu_fund               7.50%  (debt_slab)
  short_duration_fund            7.41%  (debt_slab)
  fixed_deposit                  7.03%  (debt_slab)
  gsec_retail_direct             6.93%  (debt_slab)
  recurring_deposit              6.84%  (debt_slab)


In [50]:
def best_instruments_for_need(
    need: str,
    tax_bracket: float,
    has_80c_headroom: bool,
    has_nps_headroom: bool,
    years: float,
    monthly_investment: float,
    max_risk_level: int = 10,      # add this
    top_n: int = 8
) -> pd.DataFrame:
    """
    Given a need and user's tax situation, finds the best instruments
    across ALL categories — not just within one category.
    
    This is the core of tax-aware recommendation.
    """
    
    # Define which categories are relevant for each need type
    need_categories = {
        'safe':         [AssetCategory.CASH_AND_LIQUID, 
                         AssetCategory.DEBT, 
                         AssetCategory.TAX_ADVANTAGED],
        'debt':         [AssetCategory.DEBT, 
                         AssetCategory.HYBRID,
                         AssetCategory.TAX_ADVANTAGED],
        'growth':       [AssetCategory.EQUITY, 
                         AssetCategory.HYBRID,
                         AssetCategory.TAX_ADVANTAGED],
        'gold':         [AssetCategory.GOLD_AND_COMMODITIES],
        'real_assets':  [AssetCategory.REAL_ASSETS],
        'all':          list(AssetCategory),
    }
    
    relevant_categories = need_categories.get(need, list(AssetCategory))
    
    results = []
    for ac, p in ASSET_PROFILES.items():
        # Skip if wrong category
        if p.category not in relevant_categories:
            continue
        # Skip if horizon incompatible
        if p.min_horizon_years > years:
            continue
        if p.max_horizon_years is not None and p.max_horizon_years < years:
            continue
        # Skip if minimum investment too high for monthly amount
        if p.min_investment > monthly_investment * 3:
            continue
        # Skip if too risky for this need
        if p.risk_level > max_risk_level:
            continue
        
        # Calculate effective return
        if ac == AssetClass.EQUITY_ELSS:
            eff = p.elss_effective_return(
                tax_bracket, 
                monthly_investment * 12, 
                has_80c_headroom
            )
        else:
            eff = p.effective_return_after_tax(tax_bracket, years)
        
        # Calculate what ₹1L grows to after-tax over the horizon
        projected = 100000 * (1 + eff) ** years
        
        results.append({
            'instrument':      ac.value,
            'category':        p.category.value,
            'gross_return':    p.expected_annual_return,
            'effective_return': eff,
            'projected_1L':    round(projected),
            'risk_level':      p.risk_level,
            'lock_in_yrs':     p.lock_in_years,
            'tax_treatment':   p.tax_treatment.value,
            'deduction':       p.deduction_benefit.value,
            'liquidity':       p.liquidity_score,
        })
    
    df = pd.DataFrame(results)
    df = df.sort_values('effective_return', ascending=False).head(top_n)
    df['gross_return'] = df['gross_return'].map('{:.2%}'.format)
    df['effective_return'] = df['effective_return'].map('{:.2%}'.format)
    df['projected_1L'] = df['projected_1L'].map('₹{:,.0f}'.format)
    
    return df.reset_index(drop=True)

In [51]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
pd.set_option('display.max_colwidth', 30)

print("=" * 80)
print("SAFE/DEBT INSTRUMENTS — 30% BRACKET, 3 YEAR HORIZON")
print("(hasn't used 80C limit yet)")
print("=" * 80)
df_30 = best_instruments_for_need(
    need='debt',
    tax_bracket=0.30,
    has_80c_headroom=True,
    has_nps_headroom=True,
    years=3,
    monthly_investment=10000
)
print(df_30[['instrument', 'category', 'gross_return', 
             'effective_return', 'projected_1L', 
             'risk_level', 'lock_in_yrs', 'deduction']].to_string())

print("\n")
print("=" * 80)
print("SAME NEED — 5% BRACKET, 3 YEAR HORIZON")
print("=" * 80)
df_5 = best_instruments_for_need(
    need='debt',
    tax_bracket=0.05,
    has_80c_headroom=True,
    has_nps_headroom=True,
    years=3,
    monthly_investment=10000
)
print(df_5[['instrument', 'category', 'gross_return',
            'effective_return', 'projected_1L',
            'risk_level', 'lock_in_yrs', 'deduction']].to_string())

print("\n")
print("=" * 80)
print("GROWTH INSTRUMENTS — 30% BRACKET, 7 YEAR HORIZON")
print("80C headroom available")
print("=" * 80)
df_growth = best_instruments_for_need(
    need='growth',
    tax_bracket=0.30,
    has_80c_headroom=True,
    has_nps_headroom=True,
    years=7,
    monthly_investment=10000
)
print(df_growth[['instrument', 'category', 'gross_return',
                 'effective_return', 'projected_1L',
                 'risk_level', 'lock_in_yrs', 'deduction']].to_string())

SAFE/DEBT INSTRUMENTS — 30% BRACKET, 3 YEAR HORIZON
(hasn't used 80C limit yet)
             instrument        category gross_return effective_return projected_1L  risk_level  lock_in_yrs deduction
0     aggressive_hybrid          hybrid       11.80%           10.62%     ₹135,363           5            0      none
1    balanced_advantage          hybrid       11.00%            9.90%     ₹132,737           4            0      none
2                   epf  tax_advantaged        8.20%            8.20%     ₹126,672           1            0       80c
3             nps_tier2  tax_advantaged       10.50%            7.35%     ₹123,710           4            0      none
4   conservative_hybrid          hybrid        9.00%            6.30%     ₹120,116           3            0      none
5        arbitrage_fund          hybrid        6.80%            6.12%     ₹119,507           2            0      none
6  medium_duration_fund            debt        8.30%            5.81%     ₹118,462           3

NameError: name 'elss_irr' is not defined

In [ ]:
# Diagnose why arbitrage fund isn't showing
arb = ASSET_PROFILES[AssetClass.ARBITRAGE_FUND]
print("Arbitrage fund diagnostic:")
print(f"  category: {arb.category}")
print(f"  min_horizon: {arb.min_horizon_years}")
print(f"  max_horizon: {arb.max_horizon_years}")
print(f"  min_investment: {arb.min_investment}")
print(f"  tax_treatment: {arb.tax_treatment}")

# Is it in the 'debt' need categories?
print(f"\n  AssetCategory.HYBRID in debt categories: "
      f"{AssetCategory.HYBRID in [AssetCategory.DEBT, AssetCategory.HYBRID, AssetCategory.TAX_ADVANTAGED]}")

# What's its effective return for 30% bracket, 3yr?
eff = arb.effective_return_after_tax(0.30, 3.0)
print(f"\n  Effective return 30% bracket, 3yr: {eff:.2%}")

# Diagnose ELSS
elss = ASSET_PROFILES[AssetClass.EQUITY_ELSS]
print("\nELSS diagnostic:")
print(f"  category: {elss.category}")
eff_elss = elss.elss_effective_return(0.30, 150000, True)
print(f"  Effective return 30% bracket, 80C headroom, 1.5L/yr: {eff_elss:.2%}")

Arbitrage fund diagnostic:
  category: AssetCategory.HYBRID
  min_horizon: 0.25
  max_horizon: 1.0
  min_investment: 500
  tax_treatment: TaxTreatment.EQUITY_LTCG

  AssetCategory.HYBRID in debt categories: True

  Effective return 30% bracket, 3yr: 6.12%

ELSS diagnostic:
  category: AssetCategory.EQUITY
  Effective return 30% bracket, 80C headroom, 1.5L/yr: 21.52%
